In [ ]:
from pathlib import Path
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import numpy as np 
import time

In [ ]:
### cell 0 ###

df1=pd.read_csv(f"{Path(__file__).parent.parent}/input/tmdb-movie-metadata/tmdb_5000_credits.csv")
df2=pd.read_csv(f"{Path(__file__).parent.parent}/input/tmdb-movie-metadata/tmdb_5000_movies.csv")
factor = 50
df1 = pd.concat([df1]*factor, ignore_index=True)
df1.info()

In [ ]:
### cell 1 ###

df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [ ]:
### cell 2 ###

df2.head(5)

In [ ]:
### cell 3 ###

C= df2['vote_average'].mean()
C

In [ ]:
### cell 4 ###

m= df2['vote_count'].quantile(0.9)
m

In [ ]:
### cell 5 ###

q_movies = df2.copy().loc[df2['vote_count'] >= m]
q_movies.shape

In [ ]:
### cell 6 ###

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    # Calculation based on the IMDB formula
    return (v/(v+m) * R) + (m/(m+v) * C)
# Define a new feature 'score' and calculate its value with `weighted_rating()`
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [ ]:
### cell 7 ###

#Sort movies based on score calculated above
q_movies = q_movies.sort_values('score', ascending=False)

#Print the top 15 movies
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(10)

In [ ]:
### cell 8 ###

pop = df2.sort_values('popularity', ascending=False)
import matplotlib.pyplot as plt

In [ ]:
### cell 9 ###

df2['overview'].head(5)

In [ ]:
### cell 10 ###

#Import TfIdfVectorizer from scikit-learn
df2['overview'] = df2['overview'].fillna('')

In [ ]:
### cell 11 ###

# Parse the stringified features into their corresponding python objects
from ast import literal_eval

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

In [ ]:
### cell 12 ###

# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        #Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    #Return empty list in case of missing/malformed data
    return []
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [ ]:
### cell 13 ###

# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

In [ ]:
### cell 14 ###

# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        #Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

In [ ]:
### cell 15 ###

def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])
df2['soup'] = df2.apply(create_soup, axis=1)